In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
from datetime import datetime

/Users/sahanravindu/Documents/AI ML/Noob Dev/Test Project 2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
#print(f'Open Ai Key {openai_api_key}')
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [4]:
system_message = """
You are a helpful personal finance assistant called Sahan's Finance Assistance.
Help users track their monthly salary and expenses through conversation.
Always call the right tool when the user mentions income, spending, or asks about their balance.
"""

In [ ]:
DB = 'finance.db'

def create_tables():
    with sqlite3.connect(DB) as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS income 
                     (id INTEGER PRIMARY KEY AUTOINCREMENT, 
                     amount REAL NOT NULL, month TEXT NOT NULL, 
                     year INTEGER NOT NULL, 
                     created_at TEXT NOT NULL)
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS expenses 
                     (id INTEGER PRIMARY KEY AUTOINCREMENT,
                     amount REAL NOT NULL,
                     category TEXT NOT NULL,
                     description TEXT,
                     date TEXT NOT NULL,
                     created_at TEXT NOT NULL)
        """)
        conn.commit()

create_tables()
print('Database created.')

Database created.


In [24]:
# Calcualtion methods

def set_income(amount, month, year):
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO income (amount, month, year, created_at) VALUES (?, ?, ?, ?)',
            (float(amount), month, int(year), datetime.now().isoformat())
        )
        conn.commit()
        print('New income added')
    return f'Income of ${float(amount):,.2f} for {month} {year} saved.'

def get_income():
    with sqlite3.connect(DB) as conn:
        incomes = conn.execute(
            'SELECT * FROM income'
        ).fetchall()

    if incomes is None:
        print("No income recorder")
    else:
        print(f"Lenth: {len(incomes)}")
        for income in incomes:
            print(f"Income value: {income}")

def delete_all():
    with sqlite3.connect(DB) as conn:
        conn.execute('DELETE FROM income')
        conn.commit()
    if os.path.exists(DB):
        os.remove(DB)


#delete_all()
#create_tables()
#set_income(200000.00, 'April', 2026)
#get_income()

def _get_month_context(dt=None):
    dt = dt or datetime.now()
    return dt.strftime('%B'), dt.strftime('%m'), str(dt.year), dt.year

def _query_month_totals(conn, month_num, year_str):
    salary_row = conn.execute(
        'SELECT amount FROM income WHERE month = ? AND year = ?',
        (month_name := datetime.now().strftime('%B'), int(year_str))
    ).fetchone()
    spent_row = conn.execute(
        """
        SELECT COALESCE(SUM(amount), 0) FROM expenses WHERE strftime('%m', date) = ? AND strftime('%Y', date) = ?
        """,
        (month_num, year_str)
    ).fetchone()
    return (salary_row[0] if salary_row else 0.0), spent_row[0]

def _balance_summary(month_name, year, salary, spent):
    return (f'{month_name} {year} — Salary: ${salary:,.2f} | '
            f'Spent: ${spent:,.2f} | Remaining: ${salary - spent:,.2f}')
def add_expense(amount, category, description, date=None):
    date = date or datetime.now().strftime('%Y-%m-%d')
    with sqlite3.connect(DB) as conn:
        conn.execute(
            'INSERT INTO expenses (amount, category, description, date, created_at) VALUES (?, ?, ?, ?, ?)',
            (float(amount), category, description, date, datetime.now().isoformat())
        )
        month_name, month_num, year_str, _ = _get_month_context()
        salary, spent = _query_month_totals(conn, month_num, year_str)
    return f'Logged ${float(amount):,.2f} for {category}. Spent: ${spent:,.2f} | Remaining: ${salary - spent:,.2f}'

def get_balance():
    month_name, month_num, year_str, year = _get_month_context()
    with sqlite3.connect(DB) as conn:
        salary, spent = _query_month_totals(conn, month_num, year_str)
    return _balance_summary(month_name, year, salary, spent)

def get_expense_summary():
    month_name, month_num, year_str, year = _get_month_context()
    with sqlite3.connect(DB) as conn:
        salary, _ = _query_month_totals(conn, month_num, year_str)
        rows = conn.execute(
            """
            SELECT category, SUM(amount) FROM expenses WHERE strftime('%m', date) = ? AND strftime('%Y', date) = ? GROUP BY category ORDER BY 2 DESC
            """,
            (month_num, year_str)
        ).fetchall()
    grand_total = sum(amt for _, amt in rows)
    lines = [f'{month_name} {year} Spending Breakdown:', *(
        f'  - {cat}: ${amt:,.2f} ({amt / salary * 100:.1f}% of salary)' if salary
        else f'  - {cat}: ${amt:,.2f}'
        for cat, amt in rows
    )]
    lines.append(f'Total spent: ${grand_total:,.2f} | Remaining: ${salary - grand_total:,.2f}')
    return '\n'.join(lines)

In [25]:
set_income_function = {
    'name': 'set_income',
    'description': 'Save the user monthly imcome. Call when the user mentions their income, salary, or earnings for a month.',
    'parameters': {
        'type': 'object',
        'properties': {
            'amount': {'type': 'number',  'description': 'Salary amount'},
            'month':  {'type': 'string',  'description': 'Month name e.g. April'},
            'year':   {'type': 'integer', 'description': 'Year e.g. 2026'}
        },
        'required': ['amount', 'month', 'year'],
        'additionalProperties': False
    }
}

add_expense_function = {
    'name': 'add_expense',
    'description': 'Log an expense. Call when the user mentions spending, paying, buying, or any purchase.',
    'parameters': {
        'type': 'object',
        'properties': {
            'amount':      {'type': 'number', 'description': 'Expense amount'},
            'category':    {'type': 'string', 'description': 'Category e.g. Groceries, Rent, Transport'},
            'description': {'type': 'string', 'description': 'Short description of the expense'},
            'date':        {'type': 'string', 'description': 'Date as YYYY-MM-DD. Omit to use today.'}
        },
        'required': ['amount', 'category', 'description'],
        'additionalProperties': False
    }
}

get_balance_function = {
    'name': 'get_balance',
    'description': 'Get current month balance. Call when user asks how much is left, their balance, or how they are tracking.',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': [],
        'additionalProperties': False
    }
}

get_expense_summary_function = {
    'name': 'get_expense_summary',
    'description': 'Get spending breakdown by category. Call when user asks for a summary, breakdown, or where their money went.',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': [],
        'additionalProperties': False
    }
}

tools = [
    {'type': 'function', 'function': set_income_function},
    {'type': 'function', 'function': add_expense_function},
    {'type': 'function', 'function': get_balance_function},
    {'type': 'function', 'function': get_expense_summary_function},
]

TOOL_MAP = {
    'set_salary':          set_income,
    'log_expense':         add_expense,
    'get_balance':         get_balance,
    'get_expense_summary': get_expense_summary,
}